# Import

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append("..")

from src.preprocessing import (
    convert_to_cnn2d_input, load_numpy_arrays
)
from src.model import build_cnn1d, build_cnn2d
from src.train import train_model, set_seed

#  CNN 1D

In [ ]:
dataset_path = "../data/dataset.csv"

X, y = load_dataset(dataset_path)

X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)
X_train_scaled, X_val_scaled, X_test_scaled, scaler = scale_features(X_train, X_val, X_test)

# reshape → (N, G, 1)
X_train_scaled = X_train_scaled[..., np.newaxis]
X_val_scaled = X_val_scaled[..., np.newaxis]
X_test_scaled = X_test_scaled[..., np.newaxis]

print("1D CNN train shape:", X_train_scaled.shape)

In [ ]:
model_1d = build_cnn1d(input_shape=(X_train_scaled.shape[1], 1))
model_1d, history_1d = train_model(
    model_1d,
    X_train_scaled, y_train,
    X_val_scaled, y_val,
    save_path="../models/cnn1d_best.h5",
    epochs=20,
    batch_size=64,
    monitor="val_loss",
    mode="min"
)

# CNN 2D

In [ ]:
X_responder, y_responder, X_nonresponder, y_nonresponder = load_numpy_arrays("../data/npy")

X = np.concatenate((X_responder, X_nonresponder), axis=0)
y = np.concatenate((y_responder, y_nonresponder), axis=None)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=25)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=25)

X_train = convert_to_cnn2d_input(X_train)
X_val   = convert_to_cnn2d_input(X_val)
X_test  = convert_to_cnn2d_input(X_test)

print("2D CNN train shape:", X_train.shape)
print("2D CNN val shape:", X_val.shape)
print("2D CNN test shape:", X_test.shape)

In [ ]:
model_2d = build_cnn2d(input_shape=(X_train.shape[1], X_train.shape[2], 1))
model_2d, history_2d = train_model(
    model_2d,
    X_train, y_train,
    X_val, y_val,
    save_path="../models/cnn2d_best.h5",
    epochs=20,
    batch_size=64,
    monitor="val_loss",
    mode="min"
)

# Learning curve

In [ ]:
def plot_history(history, title="Training History"):
    plt.figure(figsize=(12,4))
    # Loss
    plt.subplot(1,2,1)
    plt.plot(history.history["loss"], label="train_loss")
    plt.plot(history.history["val_loss"], label="val_loss")
    plt.title(f"{title} - Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.legend()
    # Accuracy
    if "acc" in history.history:
        plt.subplot(1,2,2)
        plt.plot(history.history["acc"], label="train_acc")
        plt.plot(history.history["val_acc"], label="val_acc")
        plt.title(f"{title} - Accuracy")
        plt.xlabel("Epoch"); plt.ylabel("Accuracy")
        plt.legend()
    plt.show()

plot_history(history_1d, "1D CNN")
plot_history(history_2d, "2D CNN")